# Level 6: Grover's Algorithm
## Qiskit Edition — Quantum Search

In this notebook, you will:

1. Understand the **Oracle + Diffusion** framework
2. Build Grover's algorithm **from scratch**
3. Search for a target in an unstructured database
4. Visualize **probability amplification** over iterations

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import numpy as np

simulator = AerSimulator()
print("Ready!")

---
## 1. The Grover Framework

**Classical search** through N items: O(N) steps
**Quantum search** (Grover): O(√N) steps — quadratic speedup!

### Two Components:
1. **Oracle** — marks the target state with a phase flip
2. **Diffusion operator** — amplifies the marked state's probability

### Optimal number of iterations:
$$k = \left\lfloor \frac{\pi}{4}\sqrt{N} \right\rfloor$$

---
## 2. Building the Oracle

The oracle flips the phase of the target state. For a target like |101⟩, we need a multi-controlled Z gate.

In [ ]:
def grover_oracle(qc, target_state, n_qubits):
    """Apply oracle that marks target_state with a phase flip.
    Uses MCZ = H on target + MCX + H on target.
    """
    # Flip qubits that should be |0⟩ in the target
    for i, bit in enumerate(reversed(target_state)):
        if bit == '0':
            qc.x(i)

    # Multi-controlled Z: H on last qubit, MCX, H on last qubit
    qc.h(n_qubits - 1)
    qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    qc.h(n_qubits - 1)

    # Undo the flips
    for i, bit in enumerate(reversed(target_state)):
        if bit == '0':
            qc.x(i)

# Demo: oracle for |101⟩
qc_demo = QuantumCircuit(3)
grover_oracle(qc_demo, '101', 3)
qc_demo.draw('mpl', style='iqp')

---
## 3. Building the Diffusion Operator

The diffusion operator reflects the state about the uniform superposition, amplifying the marked state.

In [ ]:
def diffusion_operator(qc, n_qubits):
    """Apply the Grover diffusion operator (inversion about mean)."""
    # Apply H to all qubits
    for i in range(n_qubits):
        qc.h(i)

    # Apply X to all qubits
    for i in range(n_qubits):
        qc.x(i)

    # Multi-controlled Z
    qc.h(n_qubits - 1)
    qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    qc.h(n_qubits - 1)

    # Undo X
    for i in range(n_qubits):
        qc.x(i)

    # Undo H
    for i in range(n_qubits):
        qc.h(i)

# Demo
qc_diff = QuantumCircuit(3)
diffusion_operator(qc_diff, 3)
qc_diff.draw('mpl', style='iqp')

---
## 4. Full Grover's Algorithm

In [ ]:
def grovers_algorithm(target, n_qubits, n_iterations=None):
    """Run Grover's algorithm to find the target state."""
    N = 2 ** n_qubits
    if n_iterations is None:
        n_iterations = int(np.floor(np.pi / 4 * np.sqrt(N)))

    qc = QuantumCircuit(n_qubits, n_qubits)

    # Initialize uniform superposition
    for i in range(n_qubits):
        qc.h(i)

    # Apply Grover iterations
    for _ in range(n_iterations):
        qc.barrier()
        grover_oracle(qc, target, n_qubits)
        qc.barrier()
        diffusion_operator(qc, n_qubits)

    # Measure
    qc.measure(range(n_qubits), range(n_qubits))

    return qc, n_iterations

# Search for |101⟩ in a 3-qubit space
target = '101'
qc, iters = grovers_algorithm(target, 3)
print(f"Searching for |{target}⟩ using {iters} Grover iteration(s)")

counts = simulator.run(qc, shots=2048).result().get_counts()
plot_histogram(counts, title=f"Grover's Search for |{target}⟩")

In [ ]:
# Show the full circuit
qc, _ = grovers_algorithm('101', 3)
qc.draw('mpl', style='iqp', fold=40)

---
## 5. Probability vs. Iterations

Watch how the target probability changes with each iteration.

In [ ]:
target = '101'
n_qubits = 3
max_iters = 8
shots = 4096

probabilities = []
for n_iter in range(max_iters + 1):
    qc, _ = grovers_algorithm(target, n_qubits, n_iterations=n_iter)
    counts = simulator.run(qc, shots=shots).result().get_counts()
    prob = counts.get(target, 0) / shots
    probabilities.append(prob)

plt.figure(figsize=(10, 5))
plt.bar(range(max_iters + 1), probabilities, color='steelblue', alpha=0.8)
plt.axhline(y=1/8, color='red', linestyle='--', label='Random guess (1/8)')
plt.xlabel('Number of Grover Iterations', fontsize=12)
plt.ylabel(f'Probability of |{target}⟩', fontsize=12)
plt.title('Grover\'s Algorithm: Probability Amplification', fontsize=14)
plt.xticks(range(max_iters + 1))
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nOptimal iterations for N={2**n_qubits}: {int(np.floor(np.pi/4*np.sqrt(2**n_qubits)))}")
print(f"Peak probability at iteration {np.argmax(probabilities)}: {max(probabilities):.3f}")

---
## 6. Key Takeaways

| Concept | Description |
|---------|-------------|
| **Grover's Algorithm** | Searches unstructured database in O(√N) |
| **Oracle** | Phase-flips the target state |
| **Diffusion** | Amplifies probability of marked state |
| **Optimal iterations** | $\lfloor \frac{\pi}{4}\sqrt{N} \rfloor$ |
| **Over-iteration** | Too many iterations reduces probability! |

---
## 7. Challenges

1. **Different target**: Search for |110⟩ instead of |101⟩
2. **4-qubit search**: Increase to 4 qubits and search for |1010⟩. How many iterations are optimal?
3. **Multiple targets**: Modify the oracle to mark two states simultaneously. How does this affect the number of iterations needed?

In [ ]:
# Your challenge code here!

---
**Next up: [Level 7 — Quantum Fourier Transform](../Level_07_QFT/Level_07_Qiskit.ipynb)**

# Level 6: Grover's Algorithm
## Qiskit Edition

In this notebook, you will:

1. Understand the **unstructured search** problem
2. Learn why Grover's gives an $O(\sqrt{N})$ speedup
3. Build the **oracle** and **diffusion operator** from scratch
4. Search for a specific item in an unsorted database

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import numpy as np

simulator = AerSimulator()
print("Ready!")

---
## 1. The Search Problem

**Problem**: Given an unsorted list of $N$ items, find the one marked item.

- **Classical**: Need $O(N)$ queries on average
- **Grover's**: Need only $O(\sqrt{N})$ queries!

For $N = 1{,}000{,}000$: Classical needs ~500,000 queries, Grover's needs ~1,000.

### How It Works (Geometry)

Think of the state as a vector in a 2D plane:
- **|w>**: the winning/marked state
- **|s>**: the uniform superposition of all non-winning states

Grover's algorithm **rotates** the state vector toward |w> in steps of ~$2/\sqrt{N}$ radians.

---
## 2. Building Blocks

Grover's has two components, applied iteratively:

1. **Oracle**: Flips the phase of the target state: $|w\rangle \to -|w\rangle$
2. **Diffusion operator**: Reflects about the mean amplitude (amplifies marked state)

In [ ]:
def grover_oracle(qc, target_state, n_qubits):
    """Oracle that marks a specific target state by flipping its phase.
    
    target_state: string like '101' indicating which state to mark
    """
    # Apply X to qubits that should be |0> in the target
    for i, bit in enumerate(reversed(target_state)):
        if bit == '0':
            qc.x(i)
    
    # Multi-controlled Z gate (phase flip on |1...1>)
    # MCZ = H on last qubit, then MCX, then H on last qubit
    qc.h(n_qubits - 1)
    qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    qc.h(n_qubits - 1)
    
    # Undo the X gates
    for i, bit in enumerate(reversed(target_state)):
        if bit == '0':
            qc.x(i)


def diffusion_operator(qc, n_qubits):
    """Grover diffusion operator: reflection about the mean."""
    # Apply H to all qubits
    qc.h(range(n_qubits))
    
    # Apply X to all qubits
    qc.x(range(n_qubits))
    
    # Multi-controlled Z
    qc.h(n_qubits - 1)
    qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    qc.h(n_qubits - 1)
    
    # Apply X to all qubits
    qc.x(range(n_qubits))
    
    # Apply H to all qubits
    qc.h(range(n_qubits))


print("Oracle and Diffusion defined!")

In [ ]:
def grovers_algorithm(target_state, n_qubits, n_iterations=None):
    """Full Grover's algorithm circuit."""
    if n_iterations is None:
        # Optimal number of iterations: ~pi/4 * sqrt(N)
        N = 2**n_qubits
        n_iterations = int(np.round(np.pi / 4 * np.sqrt(N)))
    
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # Step 1: Create uniform superposition
    qc.h(range(n_qubits))
    
    # Step 2: Apply Grover iterations
    for _ in range(n_iterations):
        qc.barrier()
        grover_oracle(qc, target_state, n_qubits)
        qc.barrier()
        diffusion_operator(qc, n_qubits)
    
    # Step 3: Measure
    qc.measure(range(n_qubits), range(n_qubits))
    
    return qc, n_iterations

print("Grover's algorithm defined!")

In [ ]:
# Search for |101> in 3 qubits (8 possible states)
target = "101"
qc, iters = grovers_algorithm(target, 3)

print(f"Searching for |{target}> among 8 states")
print(f"Optimal iterations: {iters}")
print(f"\nCircuit:")
qc.draw('mpl')

In [ ]:
# Run and see the result
result = simulator.run(qc, shots=1000).result().get_counts()
print(f"Results for target |{target}>:")
print(result)
plot_histogram(result, title=f"Grover: searching for |{target}>")

In [ ]:
# Try different targets
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
targets = ["000", "011", "101", "111"]

for ax, target in zip(axes, targets):
    qc, _ = grovers_algorithm(target, 3)
    result = simulator.run(qc, shots=1000).result().get_counts()
    plot_histogram(result, ax=ax, title=f"Target: |{target}>")

plt.tight_layout()
plt.show()
print("Each target is found with high probability!")

---
## 3. Effect of Number of Iterations

Too few iterations: target not amplified enough. Too many: amplitude oscillates past the target!

In [ ]:
# Show how probability changes with number of iterations
target = "101"
n_qubits = 3
probs = []

for n_iter in range(1, 8):
    qc, _ = grovers_algorithm(target, n_qubits, n_iterations=n_iter)
    result = simulator.run(qc, shots=10000).result().get_counts()
    prob = result.get(target, 0) / 10000
    probs.append(prob)

plt.figure(figsize=(10, 5))
plt.bar(range(1, 8), probs, color='steelblue')
plt.xlabel('Number of Grover iterations')
plt.ylabel(f'Probability of finding |{target}>')
plt.title('Grover: Probability vs Iterations (3 qubits)')
plt.axhline(y=1/8, color='r', linestyle='--', label='Random guess (1/8)')
plt.legend()
plt.show()
print("Too many iterations reduce the probability — it oscillates!")

---
## 4. Key Takeaways

| Concept | Description |
|---------|-------------|
| **Grover's** | Unstructured search in $O(\sqrt{N})$ |
| **Oracle** | Phase-flip the target state |
| **Diffusion** | Reflection about the mean — amplitude amplification |
| **Iterations** | Optimal = $\pi/4 \cdot \sqrt{N}$ |

---
## 5. Challenges

1. **4-qubit search**: Search among 16 items. How many iterations are optimal?
2. **Multi-target**: Mark TWO states and see how the algorithm finds either one.
3. **Amplitude plot**: Plot the amplitude of every state after each iteration to visualize the rotation.

In [ ]:
# Your challenge code here!

---
**Next up: [Level 7 — Quantum Fourier Transform](../Level_07_QFT/Level_07_Qiskit.ipynb)**